Objective: To build a multimodal regression model that predicts house prices by combining structured numerical data (square footage, bedrooms) with visual data (house images). This approach mimics how real estate experts evaluate a property—by looking at both the stats and the appearance.

Setup and Library Installation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input, concatenate, Dropout

Downloading the Dataset

In [ ]:
from google.colab import files
import os

# This will prompt you to upload the kaggle.json again to be safe
uploaded = files.upload()

# Check if the file is actually there before moving it
if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✅ Kaggle credentials set up successfully!")
else:
    print("❌ Upload failed. Please try again.")

Dataset Loading (Tabular Data)

In [ ]:
# Clean up previous attempts to ensure a fresh start
!rm -rf houses_data
!rm -f deep-learning-with-opencv-and-keras-master.zip
!rm -rf deep-learning-with-opencv-and-keras-master

# 1. Download the entire repository as a zip file from AdrianRosebrock's GitHub
# Removed -q to show wget output and potential errors
!wget https://github.com/AdrianRosebrock/deep-learning-with-opencv-and-keras/archive/refs/heads/master.zip -O deep-learning-with-opencv-and-keras-master.zip

# Check if the file was downloaded and has a reasonable size
import os
zip_file_path = 'deep-learning-with-opencv-and-keras-master.zip'

download_successful = False
if os.path.exists(zip_file_path):
    zip_size = os.path.getsize(zip_file_path)
    if zip_size > 1000: # Assuming a valid zip is at least 1KB
        print(f"Downloaded '{zip_file_path}' successfully, size: {zip_size} bytes.")
        download_successful = True
    else:
        print(f"ERROR: Downloaded zip file '{zip_file_path}' is too small ({zip_size} bytes). It might be an HTML error page or an incomplete download. Please check the wget output above.")
else:
    print(f"ERROR: Downloaded zip file '{zip_file_path}' not found. Please check the wget output above.")


if download_successful:
    # 2. Unzip the downloaded file
    !unzip -q deep-learning-with-opencv-and-keras-master.zip

    # 3. Create the target directory structure
    !mkdir -p houses_data/Houses\ Dataset

    # 4. Move the 'houses' dataset content from the unzipped repo to the desired location
    # The dataset is located in deep-learning-with-opencv-and-keras-master/datasets/houses/
    # Use -n to prevent overwriting if target exists and is newer (though rm -rf should prevent this)
    !mv deep-learning-with-opencv-and-keras-master/datasets/houses/* houses_data/Houses\ Dataset/

    # 5. Clean up the downloaded zip and the unzipped repository folder
    !rm deep-learning-with-opencv-and-keras-master.zip
    !rm -rf deep-learning-with-opencv-and-keras-master

# 6. Check the folder content
print("✅ Folder content verified:")
if os.path.exists("houses_data/Houses Dataset/"):
    print(os.listdir("houses_data/Houses Dataset/"))
else:
    print("Error: 'houses_data/Houses Dataset/' directory not found or populated.")


Loading and Verifying the Tabular Data
Objective: To load the house attributes from the text file and ensure the columns match the dataset documentation.

In [ ]:
import pandas as pd
import os

# Path to the data inside the newly cloned folder
info_path = "houses-dataset/Houses Dataset/HousesInfo.txt"

# Column names: Bedrooms, Bathrooms, Area, Zipcode, Price
cols = ["bedrooms", "bathrooms", "area", "zipcode", "price"]

# Load the data
df = pd.read_csv(info_path, sep=" ", header=None, names=cols)

print("--- Success! Data Table Preview ---")
print(df.head())
print(f"\nTotal houses in dataset: {len(df)}")

Tabular Data Preprocessing
Objective: To clean the data and scale the numerical features. Neural networks perform much better when input values are small (usually between 0 and 1) rather than large numbers like 869,500.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# 1. Initialize the Scaler
scaler = MinMaxScaler()

# 2. Scale the numerical features (Bedrooms, Bathrooms, Area)
# We use fit_transform to make these values fall between 0 and 1
X_numeric = df[["bedrooms", "bathrooms", "area"]].values
X_numeric_scaled = scaler.fit_transform(X_numeric)

# 3. Scale the target variable (Price)
# It's a common practice to scale the output for regression tasks
y = df["price"].values.reshape(-1, 1)
y_scaled = scaler.fit_transform(y)

print("--- Preprocessing Complete ---")
print("Sample of scaled input (Bedrooms, Bathrooms, Area):")
print(X_numeric_scaled[:3])

Image Loading and Preprocessing. Objective: To load the physical images from the dataset, resize them to a uniform shape, and convert them into a numerical format (arrays) that the CNN (Convolutional Neural Network) can read.

In [ ]:
import cv2

def load_house_images(df, inputPath):
    images = []
    # Get all files in the image directory once for efficiency
    all_files_in_dir = os.listdir(inputPath)

    # Create a dictionary to hold images for each house ID
    # Key: house ID (integer), Value: list of image paths for that ID
    house_images_map = {}
    for filename in all_files_in_dir:
        if filename.endswith('.jpg'):
            parts = filename.split('_')
            if len(parts) >= 2 and parts[0].isdigit():
                house_id = int(parts[0])
                if house_id not in house_images_map:
                    house_images_map[house_id] = []
                house_images_map[house_id].append(os.path.join(inputPath, filename))

    # Loop over the indexes of the houses from the DataFrame
    # df.index goes from 0 to 534, corresponding to house IDs 1 to 535
    for i in df.index:
        current_house_id = i + 1
        selected_image_path = None

        if current_house_id in house_images_map:
            # Prioritize 'frontal' images if available
            frontal_images = [p for p in house_images_map[current_house_id] if 'frontal' in os.path.basename(p)]
            if frontal_images:
                selected_image_path = frontal_images[0] # Pick the first frontal image
            else:
                # Otherwise, pick the first image found for this house ID
                selected_image_path = house_images_map[current_house_id][0]

        if selected_image_path:
            image = cv2.imread(selected_image_path)
            if image is None:
                print(f"Warning: Could not load selected image at {selected_image_path}. Skipping.")
                continue
            image = cv2.resize(image, (64, 64))
            images.append(image)
        else:
            print(f"Warning: No image found for house ID {current_house_id}. Skipping.")

    # Scale pixel intensities to the range [0, 1]
    return np.array(images) / 255.0

# Define the folder where images are located
image_folder = "houses-dataset/Houses Dataset" # Confirmed by d0120a63 output

# Load the images
print("Loading images... this may take a moment.")
X_images = load_house_images(df, image_folder)

print(f"--- Images Loaded ---")
print(f"Shape of image array: {X_images.shape}")
# (535, 64, 64, 3) means 535 images, 64x64 size, 3 color channels (RGB)


In [ ]:
import os

# Path to the image folder
image_folder = "houses-dataset/Houses Dataset"

print(f"Listing contents of: {image_folder}")
if os.path.exists(image_folder):
    contents = os.listdir(image_folder)
    if contents:
        # Print first 10 items if many, otherwise all
        print("Found files/directories:")
        for item in contents[:10]:
            print(f"- {item}")
        if len(contents) > 10:
            print(f"... and {len(contents) - 10} more items.")
    else:
        print("Directory is empty.")
else:
    print(f"Directory does not exist: {image_folder}")


Train-Test Split
Objective: To divide our data into a Training Set (which the AI uses to learn) and a Testing Set (which we use to grade the AI). We must split both the images and the tabular data identically so they still match.

In [ ]:
from sklearn.model_selection import train_test_split

# We split the data: 75% for training, 25% for testing
# We must pass both X_numeric_scaled and X_images to keep them synchronized
(trainAttrX, testAttrX, trainImagesX, testImagesX) = train_test_split(
    X_numeric_scaled, X_images, test_size=0.25, random_state=42)

# Also split the target prices (y_scaled)
(trainY, testY) = train_test_split(y_scaled, test_size=0.25, random_state=42)

print("--- Data Split Complete ---")
print(f"Training samples: {len(trainAttrX)}")
print(f"Testing samples: {len(testAttrX)}")

: Building the Multimodal Architecture (Fusion Model)
Objective: To create a "two-headed" neural network using the Keras Functional API. One head (MLP) handles the numbers, and the other head (CNN) handles the images. We then "concatenate" (join) them to make a final price prediction.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, concatenate, Dropout

# --- Branch 1: The MLP (for Tabular Data) ---
# This branch processes bedrooms, bathrooms, and area
inputs_attr = Input(shape=(3,))
x = Dense(16, activation="relu")(inputs_attr)
x = Dense(8, activation="relu")(x)
mlp_branch = Model(inputs=inputs_attr, outputs=x)

# --- Branch 2: The CNN (for Image Data) ---
# This branch processes the house photos
inputs_img = Input(shape=(64, 64, 3))
y = Conv2D(16, (3, 3), padding="same", activation="relu")(inputs_img)
y = MaxPooling2D(pool_size=(2, 2))(y)
y = Conv2D(32, (3, 3), padding="same", activation="relu")(y)
y = MaxPooling2D(pool_size=(2, 2))(y)
y = Flatten()(y)
y = Dense(16, activation="relu")(y)
cnn_branch = Model(inputs=inputs_img, outputs=y)

# --- The Fusion (Combining both branches) ---
combined = concatenate([mlp_branch.output, cnn_branch.output])

# Final layers to predict the price
z = Dense(8, activation="relu")(combined)
z = Dense(1, activation="linear")(z) # Linear activation for regression (price)

# The final model takes two inputs and produces one output
model = Model(inputs=[mlp_branch.input, cnn_branch.input], outputs=z)

# Compile the model
model.compile(loss="mse", optimizer="adam", metrics=["mae"])

print("--- Multimodal Model Built Successfully ---")
model.summary()

Model Training
Objective: To train the neural network by showing it the paired images and data. Since we have a small dataset (401 training samples), we will train for 50 epochs with a small batch size.

In [ ]:
# Train the model
print("--- Starting Training ---")
history = model.fit(
    x=[trainAttrX, trainImagesX], y=trainY,
    validation_data=([testAttrX, testImagesX], testY),
    epochs=50,
    batch_size=8
)

print("\n--- Training Complete ---")

Model Evaluation & Metrics
Objective: To evaluate the model's accuracy using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). We will also "denormalize" the predictions to see the error in actual currency (USD).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Make predictions on the test set
predictions = model.predict([testAttrX, testImagesX])

# 2. Inverse transform the predictions and actual values back to original dollar amounts
# This turns 0.123 back into something like $450,000
y_test_actual = scaler.inverse_transform(testY)
y_pred_actual = scaler.inverse_transform(predictions)

# 3. Calculate metrics
mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))

print(f"--- Final Evaluation Metrics ---")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")

# 4. Show a few comparisons
print("\n--- Sample Predictions vs Actual ---")
for i in range(5):
    print(f"Actual: ${y_test_actual[i][0]:,.2f} | Predicted: ${y_pred_actual[i][0]:,.2f}")

Visualizing Training History
Objective: To create a "Loss Curve" graph. This helps you identify if your model is overfitting (memorizing the data) or underfitting (not learning enough).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Multimodal Model Training & Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

Save the model

In [ ]:
# Save the model to a file
model.save("multimodal_house_model.h5")

print("✅ Model saved as 'multimodal_house_model.h5'")

Visualizing Model Predictions & Prototype Performance

In [ ]:
import matplotlib.pyplot as plt

# We reduce the figure size so the 64x64 images aren't stretched too thin
plt.figure(figsize=(12, 8))

for i in range(4):
    ax = plt.subplot(2, 2, i + 1)

    # 'interpolation' helps smooth out the 64x64 "blocks"
    # we use the test set images we prepared earlier
    plt.imshow(testImagesX[i], interpolation='bilinear')

    actual = y_test_actual[i][0]
    pred = y_pred_actual[i][0]
    diff = abs(actual - pred)

    # We use a clean, professional font style
    title_obj = plt.title(f"House {i+1}\nActual: ${actual:,.0f} | Predicted: ${pred:,.0f}",
                          fontsize=10, pad=10)

    # Color code the result: Green for close, Red for far off
    if diff < 150000:
        plt.setp(title_obj, color='darkgreen', fontweight='bold')
    else:
        plt.setp(title_obj, color='darkred')

    plt.axis("off")

plt.suptitle("Multimodal Prototype: Visual vs. Predicted Analysis", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

Preparing the Deployment Script (app.py)
Objective: To create a single Python file that loads your trained model and creates the user interface.

In [ ]:
# Write the Streamlit app to a file
with open('app.py', 'w') as f:
    f.write("""
import streamlit as st
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image

# 1. Load the trained model
@st.cache_resource
def load_my_model():
    return tf.keras.models.load_model('multimodal_house_model.h5')

model = load_my_model()

st.title("🏠 AI House Price Predictor")
st.write("Upload a photo and enter details to estimate the property value.")

# 2. Sidebar for Tabular Inputs
st.sidebar.header("House Specifications")
bedrooms = st.sidebar.slider("Bedrooms", 1, 10, 3)
bathrooms = st.sidebar.slider("Bathrooms", 1.0, 5.0, 2.0)
area = st.sidebar.number_input("Area (Sq Ft)", value=2000)

# 3. Image Upload
uploaded_file = st.file_uploader("Upload House Photo (Frontal View)", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Process Image
    image = Image.open(uploaded_file)
    st.image(image, caption='Uploaded Photo', use_container_width=True)

    # Prepare image for AI (64x64)
    img_array = np.array(image.convert('RGB'))
    img_resized = cv2.resize(img_array, (64, 64)) / 255.0
    img_input = np.expand_dims(img_resized, axis=0)

    # Prepare numeric data for AI (Simple scaling simulation)
    # Note: In production, use the exact scaler.transform() from your training
    numeric_input = np.array([[bedrooms, bathrooms, area]]) / [10, 5, 5000]

    if st.button("Predict Price"):
        prediction = model.predict([numeric_input, img_input])
        # Denormalize (Approximate based on your dataset range)
        final_price = prediction[0][0] * 2000000
        st.success(f"Estimated Market Value: ${final_price:,.2f}")
""")